<a href="https://colab.research.google.com/github/AarnavSawant/KVCompass/blob/main/KVCompass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KVCompass Colab Runner

This notebook is set up for the config-driven benchmark sweep path. Edit the YAML cell, run one command, then inspect the saved metrics and runtime telemetry.

In [ ]:
# Clone or refresh the repo, then install the project.
import os
from pathlib import Path

repo_dir = Path('/content/KVCompass')
if not repo_dir.exists():
    !git clone https://github.com/AarnavSawant/KVCompass.git /content/KVCompass

%cd /content/KVCompass
!git pull
!nvidia-smi
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Set HF_TOKEN from Colab secrets, then verify the login.
from google.colab import userdata
from huggingface_hub import whoami
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(whoami())


In [ ]:
# Edit this YAML to choose the benchmark scenarios, methods, and budgets you want.
from pathlib import Path

sweep_yaml = '''
sweep:
  name: colab_ruler_core
  model: Qwen/Qwen2.5-1.5B-Instruct
  device: auto
  torch_dtype: bfloat16
  methods_config_path: configs/methods.yaml
  output_dir: results/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: retrieval_long
      dataset: ruler
      data_dir: "4096"
      fraction: 0.01
      methods:
        - no_compression
        - snapkv
        - expected_attention
        - adakv_expected
        - quant_4bit
      budgets:
        no_compression: [1.0]
        default: [0.5]
    - name: memory_tight
      dataset: ruler
      data_dir: "4096"
      fraction: 0.01
      methods:
        - snapkv
        - expected_attention
        - adakv_expected
        - quant_4bit
      budgets:
        default: [0.25]
'''

config_path = Path('configs/benchmark_sweeps.colab.yaml')
config_path.write_text(sweep_yaml, encoding='utf-8')
print(config_path.read_text())


In [ ]:
# Run every method listed in the YAML with one command.
# The generic transformers message about using a dataset can still appear,
# but this benchmark path does load a Hugging Face dataset. The run groups
# requests by shared context so KVPress can reuse the compressed KV cache.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.colab.yaml


In [ ]:
# Inspect the latest sweep summary and one run directory.
import json
from pathlib import Path
import pandas as pd

summary_files = sorted(Path('results/benchmark_eval').glob('*__summary.csv'))
latest_summary = summary_files[-1]
summary_df = pd.read_csv(latest_summary)
print('SUMMARY:', latest_summary)
display(summary_df)

latest_run = max(Path('results/benchmark_eval').glob('*/*'), key=lambda p: p.stat().st_mtime).parent
print('\nLATEST RUN:', latest_run)
print('\nmetrics.json')
print((latest_run / 'metrics.json').read_text())
print('\nrun_stats.json')
print((latest_run / 'run_stats.json').read_text())


In [ ]:
# Optional: run one benchmark directly instead of the whole sweep.
!python scripts/run_kvpress_benchmark_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --method no_compression \
  --budget 1.0 \
  --fraction 0.01 \
  --torch-dtype bfloat16
